
# 練習問題: 行列積を並列化して台数効果を測る

## 目標

`#pragma omp parallel for` (Fortran では `!$omp parallel do` ... `!$omp end parallel do`) を1つ挿入するだけで,
HPC の代表的なカーネルである**密行列積** C = A * B を並列化し, スレッド数を増やすと性能 (GFLOPS) が上がる「台数効果」を測定できることを体験する.

## 課題

`matmul.cpp` (または `matmul.f90`) は, n x n の行列 A, B の積 C = A * B を 3 重ループで計算し, 実行時間と GFLOPS を表示する.
行列は 1 次元配列に格納している (A の (i,j) 要素が `A[i*n+j]`).
現状は逐次 (並列化されていない) なので, スレッド数を増やしても速くならない.

コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, **いちばん外側の i ループ** を並列化せよ.
各行 `C[i][*]` の計算は互いに独立で, k についての足し込みはループ内のローカル変数 `s` で行うので, reduction は不要である.

- C++: 計算本体の最も外側の `for` ループの直前に `#pragma omp parallel for` を1行加える.
- Fortran: 最も外側の `do` ループを `!$omp parallel do` と `!$omp end parallel do` で囲む.

それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore matmul.cpp -o matmul.exe

# Fortran
nvfortran -fast -mp=multicore matmul.f90 -o matmul.exe
```

`OMP_NUM_THREADS` を 1, 2, 4, ... と変えながら実行し, GFLOPS を比較せよ. スレッドをコアに固定するため `OMP_PROC_BIND=true` も付ける.

```
OMP_PROC_BIND=true OMP_NUM_THREADS=1 ./matmul.exe
OMP_PROC_BIND=true OMP_NUM_THREADS=2 ./matmul.exe
OMP_PROC_BIND=true OMP_NUM_THREADS=4 ./matmul.exe
```

第1引数で行列のサイズ `n` を変えられる (例: `./matmul.exe 1500`).

## 期待される結果

A, B を全要素 1.0 で初期化しているので, 答えは全要素 `C[i][j] = n` となり, `checksum = n^3`, `OK` と表示される (並列化しても結果は変わらない).

並列化後は, スレッド数を増やすと実行時間が短くなり, GFLOPS が増える (理想的にはスレッド数に比例). 例:

```
OMP_NUM_THREADS=1 ->  約  X GFLOPS
OMP_NUM_THREADS=2 -> 約 2X GFLOPS
OMP_NUM_THREADS=4 -> 約 4X GFLOPS
```

これは人工的な計算ではなく, 科学技術計算で頻出する本物の行列積カーネルである.
スレッド数を増やしても性能が頭打ちになる点も観察せよ.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ matmul.cpp
#include <cassert>
#include <cstdio>
#include <cstdlib>
#include <omp.h>

/* 密行列積 C = A * B (いずれも n x n).
   行列は 1 次元配列に格納する (A[i*n+j] が A の (i,j) 要素).
   浮動小数点演算は乗算 + 加算が n 回 / 要素なので, 全体で 2 n^3 flops. */

int main(int argc, char ** argv) {
  long n = (1 < argc ? atol(argv[1]) : 1000);
  double * A = (double *)malloc(sizeof(double) * n * n);
  double * B = (double *)malloc(sizeof(double) * n * n);
  double * C = (double *)malloc(sizeof(double) * n * n);
  assert(A && B && C);
  /* A も B も全要素 1.0 に初期化 -> C[i][j] = n になるはず (検算しやすい) */
  for (long i = 0; i < n * n; i++) { A[i] = 1.0; B[i] = 1.0; C[i] = 0.0; }
  printf("n = %ld\n", n);
  /* 計測開始 */
  double t0 = omp_get_wtime();
  /* 計算本体: 3 重ループ. C[i][j] += A[i][k] * B[k][j] */
  // TODO: いちばん外側の i ループの直前に #pragma omp parallel for を1行追加し, 行ごとに並列化せよ.
  for (long i = 0; i < n; i++) {
    for (long j = 0; j < n; j++) {
      double s = 0.0;
      for (long k = 0; k < n; k++) {
        s += A[i * n + k] * B[k * n + j];
      }
      C[i * n + j] = s;
    }
  }
  /* 計測終了 */
  double t1 = omp_get_wtime();
  double dt = t1 - t0;          /* sec */
  /* 検算: 全要素が n のはずなので, 総和は n * n * n */
  double checksum = 0.0;
  for (long i = 0; i < n * n; i++) checksum += C[i];
  double expected = (double)n * (double)n * (double)n;
  printf("checksum   : %.0f (expected %.0f) -> %s\n",
         checksum, expected, (checksum == expected ? "OK" : "NG"));
  double flops = 2. * (double)n * (double)n * (double)n;
  printf("elapsed    : %7.3f  sec\n", dt);
  printf("flops      : %.2e\n", flops);
  printf("%.3f GFLOPS\n", flops / dt * 1e-9);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore matmul.cpp -o matmul_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./matmul_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ matmul.f90
program matmul_speedup
  use omp_lib
  ! 密行列積 C = A * B (いずれも n x n).
  ! 行列は 1 次元配列に格納する (A(i*n+j+1) が A の (i,j) 要素, i,j は 0 始まり).
  ! 浮動小数点演算は乗算 + 加算が n 回 / 要素なので, 全体で 2 n^3 flops.
  character(len=32) :: arg
  integer(8) :: n, i, j, k
  real(8), allocatable :: A(:), B(:), C(:)
  real(8) :: s, t0, t1, dt, flops, checksum, expected
  n = 1000
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(A(n * n), B(n * n), C(n * n))
  ! A も B も全要素 1.0 に初期化 -> C(i,j) = n になるはず (検算しやすい)
  A = 1.0d0; B = 1.0d0; C = 0.0d0
  print "(a,i0)", "n = ", n
  ! 計測開始
  t0 = omp_get_wtime()
  ! 計算本体: 3 重ループ. C[i][j] += A[i][k] * B[k][j]
  ! TODO: いちばん外側の i ループを !$omp parallel do ... !$omp end parallel do で囲み, 行ごとに並列化せよ.
  do i = 0, n - 1
     do j = 0, n - 1
        s = 0.0d0
        do k = 0, n - 1
           s = s + A(i * n + k + 1) * B(k * n + j + 1)
        end do
        C(i * n + j + 1) = s
     end do
  end do
  ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do).
  ! 計測終了
  t1 = omp_get_wtime()
  dt = t1 - t0                  ! sec
  ! 検算: 全要素が n のはずなので, 総和は n * n * n
  checksum = sum(C)
  expected = real(n, 8) * real(n, 8) * real(n, 8)
  if (checksum == expected) then
     print "(a,f0.0,a,f0.0,a)", "checksum   : ", checksum, " (expected ", expected, ") -> OK"
  else
     print "(a,f0.0,a,f0.0,a)", "checksum   : ", checksum, " (expected ", expected, ") -> NG"
  end if
  flops = 2.d0 * real(n, 8) * real(n, 8) * real(n, 8)
  print "(a,f7.3,a)", "elapsed    : ", dt, "  sec"
  print "(a,es9.2)",  "flops      : ", flops
  print "(f7.3,a)", flops / dt * 1e-9, " GFLOPS"
end program matmul_speedup

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore matmul.f90 -o matmul_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./matmul_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:matmul.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:matmul.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:matmul.cpp}